<p style="text-align:center">
    <a href="https://skills.network" target="_blank">
    <img src="https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/assets/logos/SN_web_lightmode.png" width="200" alt="Skills Network Logo">
    </a>
</p>


<h1>实验：神经网络整流线性单元（ReLU）与 Sigmoid 对比</h1>



<h3>本笔记本目标</h3>    
<h5> 1. 定义多个神经网络、损失函数、优化器。</h5>
<h5> 2. 测试 Sigmoid 和 Relu。 </h5>
<h5> 3. 分析结果。 </h5>     


<h2>目录</h2>
<p>在本实验中，你将在 MNIST 数据集上测试 Sigmoid 和 Relu 激活函数，网络包含两个隐藏层。</p>

- [神经网络模块与训练函数](#Neural-Network-Module-and-Training-Function)
- [生成数据](#Make-Some-Data)
- [定义神经网络、损失函数、优化器并训练模型](#Define-Neural-Network,-Criterion-function,-Optimizer-and-Train-the-Model)
- [测试 Sigmoid 和 Relu](#Test-Sigmoid-and-Relu)
- [分析结果](#Analyze-Results)

<p>预计所需时间：<strong>25 分钟</strong></p>

<hr>


我们需要以下库


In [ ]:
%%time
%pip install numpy matplotlib
%pip install torch==2.8.0+cpu torchvision==0.23.0+cpu torchaudio==2.8.0+cpu \
    --index-url https://download.pytorch.org/whl/cpu

In [ ]:
# 导入本实验所需的库

# 使用以下代码安装 torchvision 库
# !conda install -y torchvision

# PyTorch 库
import torch 
# PyTorch 神经网络
import torch.nn as nn
# 允许我们转换张量
import torchvision.transforms as transforms
# 允许我们下载数据集
import torchvision.datasets as dsets
# 允许我们使用激活函数
import torch.nn.functional as F
# 用于绘制数据和损失曲线
import matplotlib.pylab as plt
# 允许我们使用数组来操作和存储数据
import numpy as np
# 设置随机种子可以控制随机性并保证结果可复现
torch.manual_seed(2)

<!--用于分隔主题的空格-->


<h2 id="Model">神经网络模块与训练函数</h2> 


定义具有两个隐藏层的神经网络模块或类


<img src="https://ibm.box.com/shared/static/5wtclahun0f70qlwkn2kxzh3amnbq4zg.png" width="200" alt="神经网络模型">


In [ ]:
# 使用 Sigmoid 作为激活函数创建模型类

class Net(nn.Module):
    
    # 构造函数
    def __init__(self, D_in, H1, H2, D_out):
        # D_in 是第一层的输入大小（输入层大小）
        # H1 是第一层的输出大小和第二层的输入大小（第一个隐藏层的大小）
        # H2 是第二层的输出大小和第三层的输入大小（第二个隐藏层的大小）
        # D_out 是第三层的输出大小（输出层大小）
        super(Net, self).__init__()
        self.linear1 = nn.Linear(D_in, H1)
        self.linear2 = nn.Linear(H1, H2)
        self.linear3 = nn.Linear(H2, D_out)
    
    # 预测
    def forward(self,x):
        # 将 x 输入第一层，然后通过 sigmoid 函数
        x = torch.sigmoid(self.linear1(x)) 
        # 将上一行的结果输入第二层，然后通过 sigmoid 函数
        x = torch.sigmoid(self.linear2(x))
        # 将上一行的结果输入第三层
        x = self.linear3(x)
        return x

定义使用 Relu 激活函数的类


In [ ]:
# 使用 Relu 作为激活函数创建模型类

class NetRelu(nn.Module):
    
    # 构造函数
    def __init__(self, D_in, H1, H2, D_out):
        # D_in 是第一层的输入大小（输入层大小）
        # H1 是第一层的输出大小和第二层的输入大小（第一个隐藏层的大小）
        # H2 是第二层的输出大小和第三层的输入大小（第二个隐藏层的大小）
        # D_out 是第三层的输出大小（输出层大小）
        super(NetRelu, self).__init__()
        self.linear1 = nn.Linear(D_in, H1)
        self.linear2 = nn.Linear(H1, H2)
        self.linear3 = nn.Linear(H2, D_out)
    
    # 预测
    def forward(self, x):
        # 将 x 输入第一层，然后通过 relu 函数
        x = torch.relu(self.linear1(x))  
        # 将上一行的结果输入第二层，然后通过 relu 函数
        x = torch.relu(self.linear2(x))
        # 将上一行的结果输入第三层
        x = self.linear3(x)
        return x

定义一个训练模型的函数，该函数返回一个 Python 字典以存储训练损失和验证数据上的准确率


In [ ]:
# 模型训练函数

def train(model, criterion, train_loader, validation_loader, optimizer, epochs=100):
    i = 0
    useful_stuff = {'training_loss': [], 'validation_accuracy': []}  
    # 在整个训练数据集上训练的次数
    for epoch in range(epochs):
        # 遍历训练加载器中的每个批次
        for i, (x, y) in enumerate(train_loader):
            # 重置计算得到的梯度值，每次都必须这样做，因为如果不重置，梯度会累积
            optimizer.zero_grad()
            # 通过将图像张量展平为 1×28*28 的张量来进行预测
            z = model(x.view(-1, 28 * 28))
            # 计算预测值与实际类别之间的损失
            loss = criterion(z, y)
            # 计算每个权重和偏置的梯度值
            loss.backward()
            # 根据计算得到的梯度值更新权重和偏置
            optimizer.step()
            # 保存损失
            useful_stuff['training_loss'].append(loss.data.item())
        
        # 用于跟踪正确预测数量的计数器
        correct = 0
        # 遍历验证数据集中的每个批次
        for x, y in validation_loader:
            # 进行预测
            z = model(x.view(-1, 28 * 28))
            # 获取具有最大值的类别
            _, label = torch.max(z, 1)
            # 检查我们的预测是否与实际类别匹配
            correct += (label == y).sum().item()
    
        # 保存百分比准确率
        accuracy = 100 * (correct / len(validation_dataset))
        useful_stuff['validation_accuracy'].append(accuracy)
    
    return useful_stuff

<!--用于分隔主题的空格-->


<h2 id="Makeup_Data">生成数据</h2> 


通过将参数 <code>train</code> 设置为 <code>True</code> 来加载训练数据集，并通过在 <code>transform</code> 参数中放置一个转换对象将其转换为张量


In [ ]:
# 创建训练数据集

train_dataset = dsets.MNIST(root='./data', train=True, download=True, transform=transforms.ToTensor())

通过将参数 <code>train</code> 设置为 <code>False</code> 来加载测试数据集，并通过在 <code>transform</code> 参数中放置一个转换对象将其转换为张量


In [ ]:
# 创建验证数据集

validation_dataset = dsets.MNIST(root='./data', train=False, download=True, transform=transforms.ToTensor())

创建损失函数


In [ ]:
# 创建损失函数

criterion = nn.CrossEntropyLoss()

创建训练数据加载器和验证数据加载器对象


In [ ]:
# 创建训练数据加载器和验证数据加载器对象

# 批量大小为 2000，shuffle=True 表示每个轮次都会对数据进行洗牌
train_loader = torch.utils.data.DataLoader(dataset=train_dataset, batch_size=2000, shuffle=True)
# 批量大小为 5000，每个轮次不会对数据进行洗牌
validation_loader = torch.utils.data.DataLoader(dataset=validation_dataset, batch_size=5000, shuffle=False)

<!--用于分隔主题的空格-->


<h2 id="Train">定义神经网络、损失函数、优化器并训练模型</h2> 


创建具有 100 个隐藏神经元的模型


In [ ]:
# 设置创建模型的参数

input_dim = 28 * 28 # Diemension of an image
hidden_dim1 = 50
hidden_dim2 = 50
output_dim = 10 # Number of classes

视频中的轮次数为 35。你现在可以尝试 10。如果尝试 35，可能需要很长时间。


In [ ]:
# 设置迭代次数

cust_epochs = 10

<!--用于分隔主题的空格-->


<h2 id="Test">测试 Sigmoid 和 Relu</h2> 


使用 Sigmoid 激活函数训练网络


In [ ]:
# 使用 sigmoid 函数训练模型

learning_rate = 0.01
# 创建 Net 模型的一个实例
model = Net(input_dim, hidden_dim1, hidden_dim2, output_dim)
# 创建一个使用学习率和梯度更新模型参数的优化器
optimizer = torch.optim.SGD(model.parameters(), lr=learning_rate)
# 训练模型
training_results = train(model, criterion, train_loader, validation_loader, optimizer, epochs=cust_epochs)

使用 Relu 激活函数训练网络


In [ ]:
# 使用 relu 函数训练模型

learning_rate = 0.01
# 创建 NetRelu 模型的一个实例
modelRelu = NetRelu(input_dim, hidden_dim1, hidden_dim2, output_dim)
# 创建一个使用学习率和梯度更新模型参数的优化器
optimizer = torch.optim.SGD(modelRelu.parameters(), lr=learning_rate)
# 训练模型
training_results_relu = train(modelRelu, criterion, train_loader, validation_loader, optimizer, epochs=cust_epochs)

<!--用于分隔主题的空格-->


<h2 id="Result">分析结果</h2> 


比较每种激活函数的训练损失


In [ ]:
# 比较训练损失

plt.plot(training_results['training_loss'], label='sigmoid')
plt.plot(training_results_relu['training_loss'], label='relu')
plt.ylabel('loss')
plt.title('training loss iterations')
plt.legend()

比较每个模型的验证损失


In [ ]:
# 比较验证损失

plt.plot(training_results['validation_accuracy'], label = 'sigmoid')
plt.plot(training_results_relu['validation_accuracy'], label = 'relu') 
plt.ylabel('validation accuracy')
plt.xlabel('Iteration')   
plt.legend()

<!--用于分隔主题的空格-->


<h2>关于作者：</h2> 

<a href="https://www.linkedin.com/in/joseph-s-50398b136/">Joseph Santarcangelo</a> 拥有电气工程博士学位，他的研究重点是利用机器学习、信号处理和计算机视觉来确定视频如何影响人类认知。Joseph 自获得博士学位以来一直在 IBM 工作。


其他贡献者：<a href="https://www.linkedin.com/in/michelleccarey/">Michelle Carey</a>、<a href="https://www.linkedin.com/in/jiahui-mavis-zhou-a4537814a">Mavis Zhou</a>



<!--## Change Log

|  Date (YYYY-MM-DD) |  Version | Changed By  |  Change Description |
|---|---|---|---|
| 2025-07-11  | 2.0  | Sathya  |  Converted lab to Jupyterlab Current|
| 2020-09-23  | 2.0  | Srishti  |  Migrated Lab to Markdown and added to course repo in GitLab |-->



<hr>

## <h3 align="center"> © IBM Corporation. All rights reserved. <h3/>
